[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [SQLModel, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlmodel-deep-dive.html)

# A Small Service


## What you will be able to do

Build the whole of it: models in a file, a database built by a migration, a FastAPI service with
five routes, and a test suite that runs against a database of its own. Every piece is one this
guide has already taught, put together in the order a project would put it. Then meet the four
mistakes that survive into real services, each one reproduced, caught by a test, and closed: a
partial update that empties the fields it did not mention, a response that publishes a column nobody
meant to publish, a list route that sends one query per row, and a test suite that passes while the
database is a migration behind the models.


## The idea

### The problem

Every notebook in this guide has shown one thing at a time, in a cell, against a database it made in
Setup. A service is not built that way. The models are a file that other files import, the database
is built by migrations rather than by `create_all`, the routes are a module, the tests are a folder,
and all of it has to keep working while people change it.

What goes wrong at that scale is not new. It is the same handful of mistakes this guide has already
named, arriving where nobody is looking at them: a `PATCH` route that takes every field of an update
model rather than the ones the caller sent, a `response_model` that is the table model, a list route
that reads a relationship in a loop, and a migration nobody wrote for a field somebody added. None
of them raises in development. Each of them is caught by one test.

### What the project is

> A **service** here is four files and a folder: **`models.py`** with the table models and the
> models that go in and out, **`database.py`** with the engine and the session dependency,
> **`main.py`** with the routes, **`migrations/`** with the revisions that build the schema, and
> **`conftest.py`** with the fixtures every test uses. The tests run with **pytest**, against a
> database made for the run and thrown away after it, with **`app.dependency_overrides`** pointing
> the service at it.

### Why it works that way

- **The models are imported, not defined in a cell.** Alembic's `env.py` imports them, `main.py`
  imports them, and the tests import them, which is why they are a module.
- **The schema comes from the migrations.** A database built by `create_all` is a database with no
  history, and the test that proves the two agree is `alembic check`.
- **The tests get their own database.** One made at the start of the run and gone at the end, so a
  test can write whatever it likes and the next one starts clean.
- **Every mistake below has a test that fails.** That is the difference between knowing about a
  mistake and being protected from it.

### Where this shows up

This is the shape of most Python services that store anything. The **Testing and Packaging** guide
is where pytest, fixtures and project layout are taught; the **APIs and JSON** guide is where FastAPI
is; the **SQLAlchemy, Deep Dive** guide's A Complete Data Layer notebook is the same exercise
without the HTTP.

### What this notebook covers

- The models, in a file
- The engine and the session, in another
- The routes: create, read, list, change and delete
- The schema, from a migration
- The tests, and the database they run against
- The service, finished
- Four failures that survive real projects, each caught by a test

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
from fastapi import Depends, FastAPI
from fastapi.testclient import TestClient
from sqlalchemy.pool import StaticPool
from sqlmodel import Field, Session, SQLModel, create_engine, select


class Hero(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str
    secret_name: str


class HeroPublic(SQLModel):
    id: int
    name: str


def get_session():
    raise RuntimeError("the application's own session, replaced in the test below")


app = FastAPI()


@app.get("/heroes", response_model=list[HeroPublic])
def read_heroes(session: Session = Depends(get_session)):
    return session.exec(select(Hero)).all()


testing = create_engine("sqlite://", poolclass=StaticPool, connect_args={"check_same_thread": False})
SQLModel.metadata.create_all(testing)
with Session(testing) as session:
    session.add(Hero(name="Deadpond", secret_name="Dive Wilson"))
    session.commit()


def override():
    with Session(testing) as session:
        yield session


app.dependency_overrides[get_session] = override
print(TestClient(app).get("/heroes").json())
```

```
[{'id': 1, 'name': 'Deadpond'}]
```

A service and its test in one screen. The application's session is whatever `get_session` returns,
and the test replaces it with one on a database it made itself, which is the whole of the fixture
this notebook builds. The rest is the same five ideas the guide has been assembling.


## Setup

Nine imports, five packages installed where they are missing, three helpers, and an empty folder for
the project.

- `sqlmodel` is the library, and the cell prints its version beside FastAPI's, Alembic's and
  pytest's. Colab has none of them pinned as this notebook wants them, so the cell installs what is
  missing, and `version` and `PackageNotFoundError`, from `importlib.metadata`, find out what that is
- `subprocess`, `sys`, `os` and `shlex` run `alembic` and `pytest` as commands and print the line
  that was run, with the project's path taken out and the time a run took removed
- `re` also takes memory addresses and pytest's own paths out of a report, `Path` names the
  project's files, and `shutil` removes the scratch folder at the start and at the end

Nothing in this notebook is imported into the notebook's own process. The service is a project on
disk, and every command that uses it runs in a Python of its own, which is what lets the files be
edited and the commands run again; a class cannot be defined twice in one session, as the
**Relationships** notebook showed.

`shell` runs a module, `failures` runs pytest and prints only what failed, and `edit` changes one
piece of a file.


In [1]:
import os
import re
import shlex
import shutil
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

for package, pin in (("sqlmodel", "sqlmodel==0.0.42"), ("fastapi", "fastapi==0.141.1"),
                     ("httpx", "httpx==0.28.1"), ("alembic", "alembic==1.20.0"), ("pytest", "pytest==8.4.2")):
    try:
        version(package)
    except PackageNotFoundError:                                    # install what this runtime is missing
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore", pin],
                       check=True)

import sqlmodel

os.environ["NO_COLOR"] = "1"                                        # no terminal codes in what a tool prints
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"                         # no stale compiled copy of a file just rewritten

SERVICE = Path("scratch/service")


def shell(*arguments, module=True):
    """Run a command in the service folder and print what it said, with the folder's own path taken out."""
    command = [sys.executable, "-m", *arguments] if module else [sys.executable, *arguments]
    done = subprocess.run(command, cwd=SERVICE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                          env={**os.environ, "COLUMNS": "80", "PYTEST_DISABLE_PLUGIN_AUTOLOAD": "1",
                               "PYTHONNODEBUGRANGES": "1", "PYTHONUNBUFFERED": "1",
                               "PYTHONWARNINGS": "ignore"})         # a warning names a file inside a library
    report = done.stdout.replace(f"{SERVICE.resolve()}{os.sep}", "")
    report = re.sub(r"\S*/_pytest/", "_pytest/", report)            # the path to pytest's own files
    report = re.sub(r"0x[0-9a-f]+", "0x...", report)                # the memory addresses of objects
    report = re.sub(r" in \d+\.\d+s\b", "", report)                 # the time a run took
    print("$", shlex.join(["python", "-m", *arguments] if module else ["python", *arguments]))
    for line in report.rstrip().splitlines():
        if not any(noise in line for noise in ("Context impl", "Will assume", "Please edit",
                                               "setting up autogenerate plugin")):
            print("   ", line)


def failures(*arguments):
    """Run pytest and print only what failed, which is what a reader needs from a report with a failure in it."""
    done = subprocess.run([sys.executable, "-m", "pytest", "--no-header", "-q", *arguments], cwd=SERVICE,
                          capture_output=True, text=True,
                          env={**os.environ, "COLUMNS": "80", "PYTEST_DISABLE_PLUGIN_AUTOLOAD": "1",
                               "PYTHONNODEBUGRANGES": "1"})
    report = re.sub(r" in \d+\.\d+s\b", "", done.stdout + done.stderr)
    for line in report.splitlines():
        if line.startswith(("E ", "FAILED", "ERROR")) or re.match(r"\d+ (passed|failed)", line.strip()):
            print(line.replace(f"{SERVICE.resolve()}{os.sep}", "")[:100])


def edit(path, old, new):
    """Change one piece of a file, and fail rather than silently do nothing."""
    text = path.read_text()
    assert text.count(old) == 1, f"{path.name}: found {text.count(old)} of {old!r}"
    path.write_text(text.replace(old, new))

shutil.rmtree("scratch", ignore_errors=True)                        # a rerun starts from no project at all
SERVICE.mkdir(parents=True)

print("sqlmodel", sqlmodel.__version__, "| fastapi", version("fastapi"), "| alembic", version("alembic"),
      "| pytest", version("pytest"))


sqlmodel 0.0.42 | fastapi 0.141.1 | alembic 1.20.0 | pytest 8.4.2


## Worked examples

### The models, in a file

Everything the service stores, and everything that goes in and out of it, in one module. The table
models carry the delete rules from the **Relationships** notebook, and the rest is the family from
the **Create, Read and Update Models** notebook:


In [2]:
%%writefile scratch/service/models.py
from sqlmodel import Field, Relationship, SQLModel


class Team(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(index=True, unique=True, max_length=50)
    headquarters: str = Field(max_length=60)

    heroes: list["Hero"] = Relationship(back_populates="team", cascade_delete=True)


class Hero(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(index=True, unique=True, max_length=50)
    secret_name: str = Field(max_length=60)                 # never leaves the service
    age: int | None = Field(default=None, index=True)
    team_id: int | None = Field(default=None, foreign_key="team.id", ondelete="CASCADE")

    team: Team | None = Relationship(back_populates="heroes")


class HeroCreate(SQLModel):
    name: str = Field(max_length=50)
    secret_name: str = Field(max_length=60)
    age: int | None = None
    team_id: int | None = None


class HeroUpdate(SQLModel):
    name: str | None = None
    secret_name: str | None = None
    age: int | None = None
    team_id: int | None = None


class TeamPublic(SQLModel):
    id: int
    name: str
    headquarters: str


class HeroPublic(SQLModel):
    id: int
    name: str
    age: int | None = None


class HeroWithTeam(HeroPublic):
    team: TeamPublic | None = None


Writing scratch/service/models.py


Seven classes, two of them tables. `secret_name` is in the table and in `HeroCreate`, because a
client supplies it, and in neither of the public models, because nothing sends it back.
`HeroUpdate` has every field optional, which is what makes a partial change possible.

### The engine and the session, in another

One module for the database, so that everything else imports the same engine and the same
dependency:


In [3]:
%%writefile scratch/service/database.py
import os

from sqlalchemy import event
from sqlalchemy.engine import Engine
from sqlmodel import Session, create_engine

DATABASE = os.environ.get("HEROES_DATABASE", "heroes.db")
engine = create_engine(f"sqlite:///{DATABASE}")


@event.listens_for(Engine, "connect")
def keep_foreign_keys(connection, record):
    """SQLite checks a foreign key only where this pragma is on, and it is off on every new connection."""
    cursor = connection.cursor()
    cursor.execute("PRAGMA foreign_keys=ON")
    cursor.close()


def get_session():
    """One session for one request, closed when the request is finished."""
    with Session(engine) as session:
        yield session


Writing scratch/service/database.py


The pragma is on the `Engine` class rather than on one engine, so that the engine a test makes gets
it too, which the `ondelete="CASCADE"` in the models depends on. The database's name comes from the
environment with a default, which is how the tests and the migrations point it somewhere else
without editing anything.

### The routes: create, read, list, change and delete

Five routes, and not one of them writes out a list of fields:


In [4]:
%%writefile scratch/service/main.py
from fastapi import Depends, FastAPI, HTTPException
from sqlalchemy.exc import IntegrityError
from sqlalchemy.orm import selectinload
from sqlmodel import Session, select

from database import get_session
from models import Hero, HeroCreate, HeroPublic, HeroUpdate, HeroWithTeam, Team

app = FastAPI(title="Heroes")


@app.post("/heroes", response_model=HeroPublic, status_code=201)
def create_hero(arriving: HeroCreate, session: Session = Depends(get_session)):
    hero = Hero.model_validate(arriving)
    session.add(hero)
    try:
        session.commit()
    except IntegrityError:
        session.rollback()
        raise HTTPException(status_code=409, detail=f"{arriving.name} is taken")
    session.refresh(hero)
    return hero


@app.get("/heroes", response_model=list[HeroWithTeam])
def list_heroes(page: int = 1, per_page: int = 5, session: Session = Depends(get_session)):
    listed = (select(Hero).order_by(Hero.name)
              .offset((page - 1) * per_page).limit(per_page)
              .options(selectinload(Hero.team)))
    return session.exec(listed).all()


@app.get("/heroes/{hero_id}", response_model=HeroWithTeam)
def read_hero(hero_id: int, session: Session = Depends(get_session)):
    hero = session.exec(select(Hero).where(Hero.id == hero_id)
                        .options(selectinload(Hero.team))).one_or_none()
    if hero is None:
        raise HTTPException(status_code=404, detail="no hero with that id")
    return hero


@app.patch("/heroes/{hero_id}", response_model=HeroPublic)
def change_hero(hero_id: int, change: HeroUpdate, session: Session = Depends(get_session)):
    hero = session.get(Hero, hero_id)
    if hero is None:
        raise HTTPException(status_code=404, detail="no hero with that id")
    hero.sqlmodel_update(change.model_dump(exclude_unset=True))      # only what the caller sent
    session.add(hero)
    session.commit()
    session.refresh(hero)
    return hero


@app.delete("/teams/{team_id}", status_code=204)
def delete_team(team_id: int, session: Session = Depends(get_session)):
    team = session.get(Team, team_id)
    if team is None:
        raise HTTPException(status_code=404, detail="no team with that id")
    session.delete(team)                                             # its heroes go with it
    session.commit()


Writing scratch/service/main.py


Five routes and five ideas, one to each: `model_validate` from the create model, a caught
`IntegrityError` as a 409, `selectinload` so that a list of heroes with their teams is two
statements, `exclude_unset` so that a change mentions only what it changes, and `cascade_delete` so
that deleting a team takes its heroes rather than leaving them pointing at nothing.

### The schema, from a migration

Alembic, pointed at the models, exactly as the **Migrations** notebook set it up:


In [5]:
shell("alembic", "init", "migrations")

ini = SERVICE / "alembic.ini"
ini.write_text(re.sub(r"^sqlalchemy\.url = .*$", "sqlalchemy.url = sqlite:///heroes.db",
                      ini.read_text(), flags=re.M))
edit(SERVICE / "migrations" / "env.py", "target_metadata = None",
     'import sys\n\nsys.path.insert(0, ".")\nfrom models import SQLModel\n\ntarget_metadata = SQLModel.metadata')
edit(SERVICE / "migrations" / "script.py.mako", "import sqlalchemy as sa",
     "import sqlalchemy as sa\nimport sqlmodel")                    # the import every revision needs

shell("alembic", "revision", "--autogenerate", "-m", "heroes and teams", "--rev-id", "0001")
shell("alembic", "upgrade", "head")


$ python -m alembic init migrations
    Creating directory migrations ...  done
    Creating directory migrations/versions ...  done
    Generating migrations/script.py.mako ...  done
    Generating migrations/env.py ...  done
    Generating migrations/README ...  done
    Generating alembic.ini ...  done
$ python -m alembic revision --autogenerate -m 'heroes and teams' --rev-id 0001
    INFO  [alembic.autogenerate.compare.tables] Detected added table 'team'
    INFO  [alembic.autogenerate.compare.constraints] Detected added index 'ix_team_name' on '('name',)'
    INFO  [alembic.autogenerate.compare.tables] Detected added table 'hero'
    INFO  [alembic.autogenerate.compare.constraints] Detected added index 'ix_hero_age' on '('age',)'
    INFO  [alembic.autogenerate.compare.constraints] Detected added index 'ix_hero_name' on '('name',)'
    Generating migrations/versions/0001_heroes_and_teams.py ...  done
$ python -m alembic upgrade head
    INFO  [alembic.runtime.migration] Running up

The template carries `import sqlmodel` from the start, so no revision in this project will ever
raise the `NameError` the **Migrations** notebook met. The database is built, and nothing has run
`create_all` against it.

### The tests, and the database they run against

`conftest.py` holds the fixtures: a database made for the run by the migrations, and a client whose
session comes from it. The `pytest.ini` beside them turns warnings off, because one of the
installations this notebook runs on warns that the HTTP client behind `TestClient` is deprecated,
and the warning names a file inside the library rather than anything in the project:


In [6]:
%%writefile scratch/service/conftest.py
import pytest
from alembic import command
from alembic.config import Config
from fastapi.testclient import TestClient
from sqlmodel import Session, create_engine

from database import get_session
from main import app


@pytest.fixture(name="engine")
def engine_fixture(tmp_path):
    """A database of this run's own, built by the migrations rather than by create_all."""
    database = tmp_path / "test.db"
    config = Config("alembic.ini")
    config.set_main_option("sqlalchemy.url", f"sqlite:///{database}")
    command.upgrade(config, "head")
    made = create_engine(f"sqlite:///{database}")
    yield made
    made.dispose()


@pytest.fixture(name="session")
def session_fixture(engine):
    """A session on that database, for a test that talks to the models directly."""
    with Session(engine) as session:
        yield session


@pytest.fixture(name="client")
def client_fixture(engine):
    """A client whose requests use that database, and an application otherwise untouched."""
    def override():
        with Session(engine) as session:
            yield session

    app.dependency_overrides[get_session] = override
    yield TestClient(app)
    app.dependency_overrides.clear()


Writing scratch/service/conftest.py


In [7]:
%%writefile scratch/service/test_service.py
from sqlmodel import select

from models import Hero, Team


def test_a_hero_is_created_and_read_back(client):
    made = client.post("/heroes", json={"name": "Deadpond", "secret_name": "Dive Wilson", "age": 30})
    assert made.status_code == 201
    hero_id = made.json()["id"]

    read = client.get(f"/heroes/{hero_id}")
    assert read.status_code == 200
    assert read.json()["name"] == "Deadpond"


def test_a_missing_hero_is_a_404(client):
    assert client.get("/heroes/999").status_code == 404


def test_a_repeated_name_is_a_409(client):
    client.post("/heroes", json={"name": "Deadpond", "secret_name": "Dive Wilson"})
    again = client.post("/heroes", json={"name": "Deadpond", "secret_name": "Someone Else"})
    assert again.status_code == 409


def test_a_body_that_is_wrong_is_a_422(client):
    refused = client.post("/heroes", json={"name": "Ghost", "secret_name": "Ana", "age": "old"})
    assert refused.status_code == 422
    assert refused.json()["detail"][0]["loc"] == ["body", "age"]


def test_the_list_is_paged(client):
    for number in range(7):
        client.post("/heroes", json={"name": f"Hero {number}", "secret_name": f"Secret {number}"})
    first = client.get("/heroes", params={"page": 1, "per_page": 5}).json()
    second = client.get("/heroes", params={"page": 2, "per_page": 5}).json()
    assert len(first) == 5
    assert len(second) == 2


def test_deleting_a_team_takes_its_heroes(client, session):
    session.add(Team(name="Preventers", headquarters="Sharp Tower"))
    session.commit()
    client.post("/heroes", json={"name": "Rusty-Man", "secret_name": "Tommy Sharp", "team_id": 1})

    assert client.delete("/teams/1").status_code == 204
    assert session.exec(select(Hero)).all() == []


Writing scratch/service/test_service.py


In [8]:
%%writefile scratch/service/pytest.ini
[pytest]
filterwarnings = ignore


Writing scratch/service/pytest.ini


In [9]:
shell("pytest", "--no-header", "-q")


$ python -m pytest --no-header -q
    ......                                                                   [100%]
    6 passed


Six tests, six passes, each on a database the migrations built and threw away. The delete test is
the one that proves `cascade_delete` and `ondelete` are doing what the models say: the team went,
and the hero that pointed at it went with it rather than being left with a `team_id` matching
nothing.

### The service, finished

The whole thing, answering requests against the database the migration built:


In [10]:
%%writefile scratch/service/try_it.py
from fastapi.testclient import TestClient

from main import app

client = TestClient(app)
client.post("/heroes", json={"name": "Deadpond", "secret_name": "Dive Wilson", "age": 30})
client.post("/heroes", json={"name": "Spider-Boy", "secret_name": "Pedro Parqueador", "age": 16})
client.patch("/heroes/2", json={"age": 17})

print("listed :", client.get("/heroes").json())
print("one    :", client.get("/heroes/1").json())
print("changed:", client.get("/heroes/2").json()["age"])
print("secret in any answer:", "secret_name" in client.get("/heroes").text)


Writing scratch/service/try_it.py


In [11]:
shell("try_it.py", module=False)


$ python try_it.py
    listed : [{'id': 1, 'name': 'Deadpond', 'age': 30, 'team': None}, {'id': 2, 'name': 'Spider-Boy', 'age': 17, 'team': None}]
    one    : {'id': 1, 'name': 'Deadpond', 'age': 30, 'team': None}
    changed: 17
    secret in any answer: False


Two heroes created, one changed by a `PATCH` that mentioned one field, a list, and no secret name
anywhere in any of it. The database is the one the migration built, and the service knows nothing
about how the tests ran.

### Where each part came from

| In the service | What it relies on | The notebook it came from |
|---|---|---|
| `models.py` | table models, the create and public family, and the delete rules | **table=True**, **Create, Read and Update Models**, **Relationships** |
| `database.py` | an engine with the foreign key pragma, and a session per request | **Engine and create_all**, **SQLModel in FastAPI** |
| `main.py` | `model_validate`, a caught `IntegrityError`, `selectinload`, `exclude_unset` | **Validation and table=True**, **Sessions**, **Loading and N+1** |
| `migrations/` | `target_metadata = SQLModel.metadata` and `import sqlmodel` in the template | **Migrations** |
| `conftest.py` | a database built by the migrations, and `dependency_overrides` | **Migrations**, **SQLModel in FastAPI** |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlmodel-deep-dive/14-a-small-service-solutions.ipynb).

**1.** Add a route that creates a team from a model with no table, answering 201 and a 409 for a
name that is taken, and a test for both.


In [12]:
# your code here


**2.** Add a route that lists teams with their heroes, and a test that the heroes are in the answer.


In [13]:
# your code here


**3.** Add `motto` to `Team`, autogenerate revision `0002`, run it, and show `alembic check` is
happy afterwards.


In [14]:
# your code here


**4.** Write a test that a `PATCH` naming only the age leaves the name alone, and run it.


In [15]:
# your code here


**5.** Write a test that a hero may not be created on a team that does not exist, run it, and say
in a comment which two pieces of the service make it pass.


In [16]:
# your code here


**6.** Run the whole suite and print how many tests there are now.


In [17]:
# your code here


## Common errors

### No error, and an age that is gone: a change without exclude_unset


In [18]:
edit(SERVICE / "main.py", "hero.sqlmodel_update(change.model_dump(exclude_unset=True))      # only what the caller sent",
     "hero.sqlmodel_update(change.model_dump())")

(SERVICE / "test_change.py").write_text('''
def test_a_change_leaves_the_other_fields_alone(client):
    client.post("/heroes", json={"name": "Deadpond", "secret_name": "Dive Wilson", "age": 30})
    client.patch("/heroes/1", json={"age": 31})

    hero = client.get("/heroes/1").json()
    assert hero["age"] == 31
    assert hero["name"] == "Deadpond"
''')

failures("test_change.py")


E       sqlite3.IntegrityError: NOT NULL constraint failed: hero.name
E       sqlalchemy.exc.IntegrityError: (sqlite3.IntegrityError) NOT NULL constraint failed: hero.nam
E       [SQL: UPDATE hero SET name=?, secret_name=?, age=? WHERE hero.id = ?]
E       [parameters: (None, None, 31, 1)]
E       (Background on this error at: https://sqlalche.me/e/20/gkpj)
FAILED test_change.py::test_a_change_leaves_the_other_fields_alone - sqlalche...
1 failed


The change mentioned an age and emptied the name, which the database refused because `name` is
`NOT NULL`, so the request became a 500 and the test could not even read the hero back. Where the
column had allowed a null it would have been worse: a 200, and a hero with no name.

`exclude_unset=True` is the whole fix, and the test is what keeps it there:


In [19]:
edit(SERVICE / "main.py", "hero.sqlmodel_update(change.model_dump())",
     "hero.sqlmodel_update(change.model_dump(exclude_unset=True))")

shell("pytest", "--no-header", "-q", "test_change.py")


$ python -m pytest --no-header -q test_change.py
    .                                                                        [100%]
    1 passed


### No error, and a secret name in every answer: the table model as the response model


In [20]:
edit(SERVICE / "main.py", '@app.get("/heroes/{hero_id}", response_model=HeroWithTeam)',
     '@app.get("/heroes/{hero_id}", response_model=Hero)')

(SERVICE / "test_secret.py").write_text('''
def test_no_answer_carries_a_secret_name(client):
    client.post("/heroes", json={"name": "Deadpond", "secret_name": "Dive Wilson"})

    for path in ("/heroes", "/heroes/1"):
        assert "secret_name" not in client.get(path).text, path
''')

failures("test_secret.py")


E           AssertionError: /heroes/1
E           assert 'secret_name' not in '{"age":null...am_id":null}'
E             
E             'secret_name' is contained here:
E               {"age":null,"id":1,"name":"Deadpond","secret_name":"Dive Wilson","team_id":null}
E             ?                                       +++++++++++
FAILED test_secret.py::test_no_answer_carries_a_secret_name - AssertionError:...
1 failed


The route answers with the table model, so every column of it is in the answer, including the one
the models file says never leaves the service. Nothing raised: the request is a 200 with a field too
many, and only a test that looks for it will say so.

A public model is the fix, and the test above is worth having in any service with a column like
that, because it checks every route at once:


In [21]:
edit(SERVICE / "main.py", '@app.get("/heroes/{hero_id}", response_model=Hero)',
     '@app.get("/heroes/{hero_id}", response_model=HeroWithTeam)')

shell("pytest", "--no-header", "-q", "test_secret.py")


$ python -m pytest --no-header -q test_secret.py
    .                                                                        [100%]
    1 passed


### No error, and a query for every hero: a list route without its loading option


In [22]:
edit(SERVICE / "main.py", ".limit(per_page)\n              .options(selectinload(Hero.team)))",
     ".limit(per_page))")

(SERVICE / "test_queries.py").write_text('''
from sqlalchemy import event

from models import Team


def test_the_list_sends_two_statements(client, session, engine):
    for number in range(5):                                 # a team each, so every hero has one to load
        session.add(Team(name=f"Team {number}", headquarters=f"Base {number}"))
    session.commit()
    for number in range(5):
        client.post("/heroes", json={"name": f"Hero {number}", "secret_name": f"Secret {number}",
                                     "team_id": number + 1})

    counted = []
    event.listen(engine, "before_cursor_execute",
                 lambda connection, cursor, statement, *rest: counted.append(statement))
    client.get("/heroes")

    assert len([statement for statement in counted if statement.startswith("SELECT")]) <= 2, counted
''')

failures("test_queries.py")


E       AssertionError: ['SELECT hero.id, hero.name, hero.secret_name, hero.age, hero.team_id 
E         FROM hero ORDER BY hero.name
E          LIMIT ? OFFSET ...CT team.id AS team_id, team.name AS team_name, team.headquarters AS tea
E         FROM team 
E         WHERE team.id = ?']
E       assert 6 <= 2
E        +  where 6 = len(['SELECT hero.id, hero.name, hero.secret_name, hero.age, hero.team_id \nFR
FAILED test_queries.py::test_the_list_sends_two_statements - AssertionError: ...
1 failed


Six `SELECT` statements for five heroes: one for the page and one for each hero's team, sent while
the answer was being built. The route still works, the answer is still right, and the cost grows
with the size of the page, which is the **Loading and N+1** notebook's subject arriving in a
service.

The option goes back, and the test stays:


In [23]:
edit(SERVICE / "main.py", ".limit(per_page))",
     ".limit(per_page)\n              .options(selectinload(Hero.team)))")

shell("pytest", "--no-header", "-q", "test_queries.py")


$ python -m pytest --no-header -q test_queries.py
    .                                                                        [100%]
    1 passed


### alembic.util.exc.AutogenerateDiffsDetected: New upgrade operations detected


In [24]:
edit(SERVICE / "models.py", "    age: int | None = Field(default=None, index=True)",
     "    age: int | None = Field(default=None, index=True)\n"
     "    catchphrase: str | None = Field(default=None, max_length=60)")

(SERVICE / "test_migrations.py").write_text('''
from alembic import command
from alembic.config import Config


def test_the_migrations_describe_the_models(engine):
    """alembic check: what the models say and what the migrations build are the same schema."""
    config = Config("alembic.ini")
    config.set_main_option("sqlalchemy.url", str(engine.url))
    command.check(config)
''')

failures("test_migrations.py")


E           alembic.util.exc.AutogenerateDiffsDetected: New upgrade operations detected: [('add_colu
FAILED test_migrations.py::test_the_migrations_describe_the_models - alembic....
1 failed


The models gained a column and the migrations did not, which every other test in the suite is
perfectly happy about: they build their database from the migrations and never notice the field is
missing, because nothing they do touches it. The service would notice on the first request that
mentioned it, in production.

`alembic check` is the test that catches it, and the fix is the revision that should have been
written:


In [25]:
shell("alembic", "revision", "--autogenerate", "-m", "a catchphrase", "--rev-id", "0002")
shell("alembic", "upgrade", "head")
shell("pytest", "--no-header", "-q")


$ python -m alembic revision --autogenerate -m 'a catchphrase' --rev-id 0002
    INFO  [alembic.autogenerate.compare.tables] Detected added column 'hero.catchphrase'
    Generating migrations/versions/0002_a_catchphrase.py ...  done
$ python -m alembic upgrade head
    INFO  [alembic.runtime.migration] Running upgrade 0001 -> 0002, a catchphrase
$ python -m pytest --no-header -q
    ..........                                                               [100%]
    10 passed


Every test passes, including the one that compares the models with the migrations, and the service
has a column it can use. That test is the one to write first in any project that has both models and
migrations, because it is the only one that fails for a mistake nobody made on purpose.

Last, this cell removes the scratch folder with the service, its migrations and its databases in it:


In [26]:
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


## Recap

- A service is files: models, the engine and session, the routes, the migrations, and the fixtures
  the tests share. Nothing in it is defined in a cell.
- The schema comes from migrations, and `alembic check` in a test is what keeps the models and the
  migrations describing the same thing.
- Tests get a database built for the run and thrown away after it, with `dependency_overrides`
  pointing the service at it and the application otherwise untouched.
- `exclude_unset=True`, a public response model, and `selectinload` in a list route are three lines
  that each close a failure nothing else reports.
- Every one of those failures is caught by a test that is four lines long, which is the difference
  between knowing about a mistake and being protected from it.


## What is next

That is the guide. You can write a model that is a table and a schema at once, put a session behind
a request, keep what a client sends and what it sees apart, say what happens to the children when a
parent is deleted, count the queries a page sends, change a schema that has rows in it, and test all
of it against a database of the run's own.

Where to go from here depends on what you are building. The **SQLAlchemy, Deep Dive** guide is
underneath everything here, and it is where the harder questions go: Core, transactions, loading
strategies in full, and four databases from one codebase. The **APIs and JSON** guide is the other
half of the service, from status codes to hosting. And the guides on PostgreSQL and on documents are
where the same models meet a database that is not on your laptop.


---

&#8592; **Previous:** [SQLModel in FastAPI](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlmodel-deep-dive/13-sqlmodel-in-fastapi.ipynb)  &nbsp;·&nbsp;  [SQLModel, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlmodel-deep-dive.html)
